# DataDome: from a customer's fraud ticket to a better detection signal - analysis notebook
Independent case study based on public data, not affiliated with or endorsed by DataDome.
PUBLIC DATA = RBA login dataset (synthesized, Feb 2020 - Feb 2021) and web robot sessions (March 2018).
SIMULATED = a fictional anti-fraud platform with 12 fictional customers (seed 42). Outputs are short on purpose; the full
tables are in `data/output/`. Run `python run_all.py --skip-download` first.

In [1]:
import json, duckdb, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'data' / 'output'
r = lambda n: pd.read_csv(OUT / f'{n}.csv')
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 12)

## 1. Sources (read on 26 Nov 2026)

In [2]:
f = json.loads((ROOT / 'data/public/sources_facts.json').read_text(encoding='utf-8'))
print(f['job']['title'], '|', f['job']['departments'], '| first published', f['job']['first_published'][:10])
print('SIRENE:', f['sirene']['name'], f['sirene']['siren'], '| accounts', f['sirene']['accounts_year'])
print('reference lists:', {k: v for k, v in f['reference_lists'].items() if k.endswith(('patterns', 'prefixes', 'ips'))})

Fraud Analyst | ['Threat Research'] | first published 2026-10-13
SIRENE: DATADOME 811474949 | accounts 2024
reference lists: {'crawler_ua_patterns': 1510, 'googlebot_prefixes': 317, 'gptbot_prefixes': 18, 'aws_prefixes': 10777, 'tor_exit_ips': 1259}


## 2. RBA profile (PUBLIC DATA, synthesized values)

In [3]:
r('rba_calibration').T

,0
logins,31269264
users,4304857
first_day,2020-02-03
last_day,2021-02-28
human_success_rate,0.4166
attack_success_rate,0.2598
attack_ip_share,0.099
top10_asn_share_of_attacks,0.8701
ato_logins,141
ato_per_million,4.51


In [4]:
r('rba_monthly')[['month', 'logins', 'attack_ip_share', 'ato_logins']].tail(6)   # no takeover label after Nov 2020

,month,logins,attack_ip_share,ato_logins
7,2020-09-01,2162510,0.0975,16
8,2020-10-01,2313404,0.0961,14
9,2020-11-01,3090246,0.0865,8
10,2020-12-01,3270165,0.1074,0
11,2021-01-01,3100885,0.1066,0
12,2021-02-01,2203240,0.1256,0


## 3. As-of signals and rules with a time split (train Feb-Aug 2020, test Sep-Nov 2020)

In [5]:
r('rba_signal_rates')

,is_ato,logins,new_country,new_browser,new_device_type,ip_fail_1h_ge1,ip_logins_1h_ge5,attack_ip
0,False,12541302,0.0050,0.2777,0.0583,0.2246,0.0445,0.0641
1,True,140,0.4786,0.4500,0.1571,0.1000,0.0143,0.5429


In [6]:
r('rba_rules').query("period == 'test'")[['rule', 'precision', 'recall', 'alerts_per_day', 'false_alerts_per_ato']]

,rule,precision,recall,alerts_per_day,false_alerts_per_ato
1,R1 new country for the user,0.00154,0.579,157.0,648.0
3,R2 new browser version for the user,0.00002,0.553,10774.0,46687.0
5,R3 failed logins from the IP (1 h),0.00000,0.000,448.0,NaN
7,R4 logins from the IP (1 h),0.00000,0.000,332.0,NaN
9,R5 new country AND new browser,0.00188,0.526,117.0,531.0
11,R6 IP on the attack-IP list,0.00010,0.474,1978.0,9997.0


In [7]:
r('rba_topk')

,k,ato_in_top_k,precision_at_k,ato_in_test,logins_in_test,share_reviewed
0,100,3,0.0300,38,2950315,0.000034
1,1000,14,0.0140,38,2950315,0.000339
2,10000,21,0.0021,38,2950315,0.003389
3,100000,24,0.0002,38,2950315,0.033895


Leakage check (SIMULATED, E7): a model that reads the account table extracted after the facts looks perfect; signals
known at login time with a time split give the honest number.

In [8]:
r('e7_auc')[['model', 'evaluated_on', 'auc', 'precision_at_alert', 'recall_at_alert', 'alerts_per_day']]

,model,evaluated_on,auc,precision_at_alert,recall_at_alert,alerts_per_day
0,A leaky: account locked in the 23 Nov snapshot,all rows (no split),0.998,0.0668,1.000,40.0
1,"B existing signals, as of login",test (12 Oct - 22 Nov),0.872,0.0012,0.982,2256.0
2,"C existing + S1 device cluster, as of login",test (12 Oct - 22 Nov),0.994,0.0203,0.973,129.0


## 4. Behavioural bot signals (PUBLIC DATA, March 2018, label as given by the dataset)

In [9]:
r('webrobots_feature_auc').head(6)[['feature', 'auc', 'higher_for']]

,feature,auc,higher_for
0,UNASSIGNED,0.9423,robots
1,SF_REFERRER,0.8974,humans
2,NUMBER_OF_REQUESTS,0.8297,humans
3,WIDTH,0.8229,humans
4,DATA,0.8055,humans
5,SF_FILETYPE,0.8045,humans


In [10]:
r('webrobots_models')

,model,auc_mean,auc_fold_sd,recall_at_1pct_fpr,features
0,logistic regression,0.9854,0.0011,0.779,all 35
1,gradient boosting,0.9960,0.0006,0.925,all 35
2,logistic regression,0.8866,0.0032,0.164,lab subset 11
3,gradient boosting,0.9579,0.0006,0.381,lab subset 11


In [11]:
r('lab_scores')[['visitor', 'kind', 'NUMBER_OF_REQUESTS', 'robot_score']]   # local lab, 127.0.0.1 only

,visitor,kind,NUMBER_OF_REQUESTS,robot_score
0,human-1,human,20,0.000
1,human-2,human,28,0.000
2,human-3,human,24,0.000
3,bot-fast-reader,robot,60,0.966
4,bot-slow-reader,robot,24,0.907
5,bot-login-burst,robot,80,0.546


## 5. Monitoring of the simulated platform (SIMULATED)

In [12]:
con = duckdb.connect()
con.sql(f"""SELECT customer_id, COUNT(*) AS raw_events, COUNT(DISTINCT event_id) AS unique_events
           FROM read_parquet('{(ROOT / 'data/sim/login_events/*/*.parquet').as_posix()}', hive_partitioning = true)
           GROUP BY 1 ORDER BY 1""").df()

,customer_id,raw_events,unique_events
0,C01,421983,421983
1,C02,289988,289988
2,C03,884756,884756
3,C04,436980,436980
4,C07,359287,337667
5,C10,242879,242879


In [13]:
r('alerts')[['week', 'customer_id', 'status']]

,week,customer_id,status
0,2026-09-21,C05,alert
1,2026-10-05,C03,alert
2,2026-10-12,C03,alert
3,2026-10-12,C07,data quality
4,2026-10-19,C07,data quality
5,2026-10-26,C07,data quality
6,2026-11-02,C04,expected after rule R12 (review its effect)
7,2026-11-09,C04,expected after rule R12 (review its effect)
8,2026-11-16,C01,explained by the customer's campaign
9,2026-11-16,C04,expected after rule R12 (review its effect)


## 6. The 8 events

In [14]:
r('tests')[['test', 'passed']]

,test,passed
0,K01 login event_id unique after deduplication,True
1,K02 sign-up event_id unique,True
2,K03 crawler session_id unique,True
3,Q01 duplicates exist only in SDK 4.2 events,True
4,Q02 traffic_daily login requests = raw login e...,True
5,Q03 traffic_daily sign-up requests = sign-up e...,True
6,Q04 traffic_daily verified_bot requests = craw...,True
7,P01 no IP address in published tables (hash of...,True
8,P02 no original RBA User ID in outputs (no use...,True
9,P03 no published group under 5 accounts,True


In [15]:
print(r('e1_summary').T)
print(r('e1_ip_rate').T)

                          0
customer_id             C03
cluster_devices          25
attempts             400278
ip_device_pairs      192323
failure_rate         0.9742
first_day        2026-10-05
last_day         2026-10-18
ua_family        Chrome 109
                                         0
distinct_ips                     9400.0000
max_attempts_per_ip_hour            1.0000
avg_attempts_per_active_ip_hour     1.0000
share_recommended_allow             0.8593


In [16]:
r('e3_summary').T

,0
customer_id,C05
devices,40
signups,3800
disposable_share,0.9139
prepaid_share,0.9545
avg_username_entropy,1.6
trial_conversion,0.0
share_recommended_allow,0.9487
first_day,2026-09-14
last_day,2026-10-11


In [17]:
r('e4_crawlers').head(3)

,customer_id,claimed_bot,ip_in_published_ranges,ip_type,sessions,requests,ips,allowed_share
0,C08,Googlebot,False,datacenter,3114,1218262,342,1.0
1,C08,Googlebot,True,search_or_ai_provider,2579,540993,1994,1.0
2,C10,Googlebot,True,search_or_ai_provider,2551,537115,1990,1.0


In [18]:
r('e5_dedup_effect').tail(4)

,week,logins_raw,deny_rate_raw,empty_device_share,logins_clean,deny_rate_clean
8,2026-10-26,31829,0.0690,0.2013,27924,0.0783
9,2026-11-02,32146,0.0726,0.2009,28259,0.0824
10,2026-11-09,32500,0.0708,0.2023,28461,0.0805
11,2026-11-16,31662,0.0737,0.1974,27812,0.0836


In [19]:
r('e6_simpson')

,period,channel,challenges,pass_rate,share_of_challenges
0,1 before R12 (5-25 Oct),total,2497,0.5202,0.5000
1,1 before R12 (5-25 Oct),app,394,0.9010,0.0789
2,1 before R12 (5-25 Oct),web,2103,0.4489,0.4211
3,2 after R12 (2-22 Nov),total,4995,0.6821,0.5000
4,2 after R12 (2-22 Nov),app,2849,0.8929,0.2852
5,2 after R12 (2-22 Nov),web,2146,0.4021,0.2148


## 7. Customer case T07

In [20]:
r('e2_summary').T

,0
accounts_with_success_from_cluster,9930
accounts_taken_over,214
taken_over_and_reported,214
first_takeover,2026-10-05
last_takeover,2026-10-18
reported_before_ticket,27
share_recommended_allow,1.0


In [21]:
print((ROOT / 'docs/customer_case_report_T07.md').read_text(encoding='utf-8')[:900])

# Customer case report - ticket T07 (customer C03, streaming) - SIMULATED

Ticket received 2026-10-20: "Several customers report that the e-mail address of their account was changed without their action. Are we under attack?" Priority: high. Report drafted on 2026-10-21 from data up to 2026-10-20; counts updated with the 23 Nov extract.

## Summary (5 lines)

1. Yes: C03 was the target of a credential stuffing campaign from 2026-10-05 to 2026-10-18 (400,278 login attempts, 97.4% failed).
2. The attempts came from 9,400 residential and mobile IPs, never more than 1 per IP and per hour, so the per-IP limit (5 per hour) never triggered.
3. They shared only 25 device fingerprints and an old browser version (Chrome 109): that is how the campaign is recognised.
4. 9,930 accounts had a successful login from these devices; 214 of them had their e-mail address changed within the hour (account tak


## 8. Signal proposals, measured offline (SIMULATED)

In [22]:
r('signal_offline')[['signal', 'alerts_per_day', 'precision', 'recall', 'baseline_alerts_per_day']]

,signal,alerts_per_day,precision,recall,baseline_alerts_per_day
0,S1 device cluster (added to existing signals),129.0,0.0203,0.973,2256.0
1,S2 success then credential change within 1 h,42.0,0.0628,0.991,NaN
2,S3 crawler claim not verified by published IP ...,54.0,NaN,NaN,NaN
